# Chapter 5 — GELU & Residual Connections

> Course: **llm.c — Zero to Hero**, Chapter 5 of ~20.
> Builds on Chapters 1–4 (forward/backward pattern, scatter-add, OpenMP).

After the heavy machinery of LayerNorm and matmul, this is a deliberately lighter chapter. Both ops in this chapter are **pure elementwise**: one input element produces one output element, with no reductions, no neighbor reads, and no shared state. They're the simplest layers in a Transformer.

But "simple" doesn't mean "uninteresting". GELU teaches us:

- That the GPT-2 GELU is an **approximation** — and *which* approximation matters because that's what the published weights were trained with.
- That a 1-line forward can have a **5-term backward** (and you'll write that backward by hand).
- That elementwise ops with **no inter-element dependency** are the *perfect* mental model for what GPU threads do best — foreshadowing Chapters 9–11.

Residuals teach us one big idea: **gradients of an `out = a + b` op flow unchanged to both inputs**. That's not just a coding detail — it's the reason 100-layer Transformers train at all.

### Learning objectives

By the end of this chapter you will:

- State the exact and tanh-approximate GELU formulas and explain why GPT-2 uses the latter.
- Read and write `gelu_forward` and `gelu_backward` in C; derive the backward by hand.
- Read and write `residual_forward` and `residual_backward` (both 2-line functions) and explain the **gradient-splitting** property.
- Recognize *why* element-wise ops are the easiest things to put on a GPU.


## 1. GELU — Concept

The Gaussian Error Linear Unit (Hendrycks & Gimpel, 2016) is a smooth nonlinearity:

$$\text{GELU}(x) = x \cdot \Phi(x) = \frac{x}{2}\Big(1 + \text{erf}\big(\tfrac{x}{\sqrt{2}}\big)\Big)$$

where $\Phi$ is the standard Normal CDF. Intuitively: multiply $x$ by *the probability that a standard normal is less than $x$*. So very negative inputs get squashed near 0 (because $\Phi(x) \approx 0$), very positive inputs pass through almost unchanged ($\Phi(x) \approx 1$), and around $x=0$ the curve is smooth — unlike ReLU's kink.

### The tanh approximation — and why GPT-2 uses it

`erf` is moderately expensive, so the original GELU paper also gave a tanh-based approximation:

$$\text{GELU}(x) \approx \frac{x}{2}\Big(1 + \tanh\!\Big(\sqrt{\tfrac{2}{\pi}}\,(x + 0.044715\, x^3)\Big)\Big)$$

GPT-2 was **trained with this approximation**, so any code that wants to load GPT-2 weights and produce matching outputs must also use this exact formula. PyTorch makes this explicit via `nn.GELU(approximate='tanh')`. Using `nn.GELU()` (the exact erf form) would silently produce slightly different activations, slightly different losses, and you'd lose bit-equivalence with the published model.

`llm.c` hardcodes the tanh form. The constants `0.044715` and `sqrt(2/π) ≈ 0.7978845` come straight from the paper.


## 2. PyTorch Baseline

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)
x = torch.linspace(-3, 3, 9)
gelu_exact = nn.GELU(approximate='none')(x)
gelu_tanh  = nn.GELU(approximate='tanh')(x)

print("x          ", x.tolist())
print("erf  GELU  ", [round(v, 4) for v in gelu_exact.tolist()])
print("tanh GELU  ", [round(v, 4) for v in gelu_tanh.tolist()])
print("max diff between the two:", (gelu_exact - gelu_tanh).abs().max().item())


The two are very close (diff ≲ 1e-4) but not identical. GPT-2 was trained with `approximate='tanh'`, so that's the one we'll match.


## 3. The C Forward Pass

From [`train_gpt2.c`](train_gpt2.c) lines 405–415:

```c
#define GELU_SCALING_FACTOR sqrtf(2.0f / M_PI)

void gelu_forward(float* out, float* inp, int N) {
    // (approximate) GeLU elementwise non-linearity in the MLP block of Transformer
    for (int i = 0; i < N; i++) {
        float x = inp[i];
        float cube = 0.044715f * x * x * x;
        out[i] = 0.5f * x * (1.0f + tanhf(GELU_SCALING_FACTOR * (x + cube)));
    }
}
```

Three observations:

1. **No `B`, `T`, `C` arguments.** Just a flat length `N`. Because GELU is elementwise, the layer doesn't care about the tensor's shape — every element is independent. Whoever calls this passes `N = B*T*C` (or whatever the activation size is).
2. **`GELU_SCALING_FACTOR = sqrt(2/π)`** is computed once at compile time (well, *almost* — `sqrtf` is technically a function call, but `M_PI` is a constant macro, so most compilers fold this).
3. **No `#pragma omp parallel for`.** This loop is embarrassingly parallel — it would be *trivial* to add OpenMP — but the cost of GELU is so small relative to a matmul that `llm.c` doesn't bother. A different implementation choice on the GPU side: the CUDA version *will* parallelize this, because GPU threads cost essentially nothing and the op is bandwidth-bound.


## 4. Translation Bridge

| PyTorch | C in `llm.c` |
|---|---|
| `nn.GELU(approximate='tanh')` | The 4-line `gelu_forward` above |
| `nn.GELU()` (default = exact erf) | **Not used** — GPT-2 was trained on tanh form |
| Layer applies to any shape | C function takes flat `N`; caller flattens any tensor |
| Autograd handles backward | You write `gelu_backward` by hand |

Mental model: a 1-D elementwise op like GELU is just **`for i in 0..N: out[i] = f(inp[i])`**. Shape is irrelevant. This is the same shape that GPU element-wise kernels take — one thread per element.


## 5. Compile and Verify Forward

In [ ]:
!mkdir -p course/ch05_build


In [ ]:
%%writefile course/ch05_build/gelu_forward.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

#define GELU_SCALING_FACTOR sqrtf(2.0f / M_PI)

void gelu_forward(float* out, float* inp, int N) {
    for (int i = 0; i < N; i++) {
        float x = inp[i];
        float cube = 0.044715f * x * x * x;
        out[i] = 0.5f * x * (1.0f + tanhf(GELU_SCALING_FACTOR * (x + cube)));
    }
}

static void* rd(const char* p, size_t n) {
    FILE* f = fopen(p, "rb"); if (!f){perror(p); exit(1);}
    void* b = malloc(n); size_t r = fread(b,1,n,f); (void)r; fclose(f); return b;
}

int main(int argc, char** argv) {
    if (argc != 2) return 1;
    int N = atoi(argv[1]);
    float* inp = (float*) rd("course/ch05_build/inp.bin", (size_t)N*sizeof(float));
    float* out = (float*) malloc((size_t)N*sizeof(float));
    gelu_forward(out, inp, N);
    FILE* f = fopen("course/ch05_build/out.bin","wb"); fwrite(out,4,(size_t)N,f); fclose(f);
    free(inp); free(out); return 0;
}


In [ ]:
!gcc -O3 -Wall -o course/ch05_build/gelu_forward course/ch05_build/gelu_forward.c -lm


In [ ]:
import numpy as np, torch, torch.nn as nn, subprocess
torch.manual_seed(0)
N = 1000
x = torch.randn(N) * 3.0     # range up to ~9 sigma
x.numpy().astype(np.float32).tofile("course/ch05_build/inp.bin")
subprocess.run(["./course/ch05_build/gelu_forward", str(N)], check=True)
out_c = np.fromfile("course/ch05_build/out.bin", dtype=np.float32)
out_pt = nn.GELU(approximate='tanh')(x).numpy()
print(f"GELU forward max diff: {np.max(np.abs(out_c - out_pt)):.2e}")


## 6. GELU Backward — Math

Let $g(x) = \sqrt{2/\pi}\,(x + 0.044715\,x^3)$ so that
$$\text{GELU}(x) = \tfrac{1}{2}x\,(1 + \tanh g(x))$$

Differentiating with the product rule:

$$\frac{d\,\text{GELU}}{dx} = \tfrac{1}{2}(1 + \tanh g(x)) + \tfrac{1}{2}x\cdot \text{sech}^2(g(x))\cdot g'(x)$$

with

$$g'(x) = \sqrt{2/\pi}\,(1 + 3\cdot 0.044715\, x^2)$$

That's everything — three terms (`tanh`, `sech²`, the polynomial `g'`), and a single elementwise multiply by the upstream gradient at the end:

$$\frac{\partial L}{\partial x} = \frac{d\,\text{GELU}}{dx}\cdot \frac{\partial L}{\partial \text{out}}$$

### The C version

From [`train_gpt2.c`](train_gpt2.c) lines 422–433:

```c
#pragma float_control(precise, on, push)             // see "compiler curio" below
#if defined(__GNUC__) && !defined(__clang__)
__attribute__((optimize("no-finite-math-only")))
#endif
void gelu_backward(float* dinp, float* inp, float* dout, int N) {
    for (int i = 0; i < N; i++) {
        float x = inp[i];
        float cube = 0.044715f * x * x * x;
        float tanh_arg  = GELU_SCALING_FACTOR * (x + cube);
        float tanh_out  = tanhf(tanh_arg);
        float coshf_out = coshf(tanh_arg);
        float sech_out  = 1.0f / (coshf_out * coshf_out);     // sech^2
        float local_grad = 0.5f * (1.0f + tanh_out)
                         + x * 0.5f * sech_out * GELU_SCALING_FACTOR
                                    * (1.0f + 3.0f * 0.044715f * x * x);
        dinp[i] += local_grad * dout[i];
    }
}
#pragma float_control(pop)
```

Two non-obvious bits worth pointing at:

- **`sech²` via `1.0f / (cosh * cosh)`.** There's no `sechf` in `<math.h>`, so we compute it from `cosh`. For very large `|tanh_arg|`, `cosh` overflows to `+inf`, and `1.0f / (inf * inf) = 0.0f` — which is the *correct* limit (the gradient should saturate to 0 in the tails).
- **`+=` not `=`.** As before, the same `dinp` buffer is shared with the residual path.

### Compiler curio (skim if you don't care)

The `#pragma float_control(precise, on, push)` and the GCC `__attribute__((optimize("no-finite-math-only")))` are **a deliberate de-optimization**, present to fix [issue #168](https://github.com/karpathy/llm.c/issues/168). The repo compiles with `-Ofast`, which implies `-ffinite-math-only` — telling the compiler "you can assume no `inf` or `NaN` ever happens". But here `cosh(tanh_arg)` legitimately overflows to `+inf` for large `|x|`. With `-ffinite-math-only` enabled, the compiler is allowed to fold `1.0f / (inf * inf)` into something other than `0`, and the gradient becomes wrong. The pragma turns that optimization off **just for this function**. Welcome to C compilation.


## 7. Compile and Verify GELU Backward Against Autograd

In [ ]:
%%writefile course/ch05_build/gelu_backward.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

#define GELU_SCALING_FACTOR sqrtf(2.0f / M_PI)

// Use -O2 (not -Ofast) so we don't hit the finite-math issue.
void gelu_backward(float* dinp, float* inp, float* dout, int N) {
    for (int i = 0; i < N; i++) {
        float x = inp[i];
        float cube = 0.044715f * x * x * x;
        float tanh_arg  = GELU_SCALING_FACTOR * (x + cube);
        float tanh_out  = tanhf(tanh_arg);
        float coshf_out = coshf(tanh_arg);
        float sech_out  = 1.0f / (coshf_out * coshf_out);
        float local_grad = 0.5f * (1.0f + tanh_out)
                         + x * 0.5f * sech_out * GELU_SCALING_FACTOR
                                    * (1.0f + 3.0f * 0.044715f * x * x);
        dinp[i] += local_grad * dout[i];
    }
}

static void* rd(const char* p, size_t n) {
    FILE* f = fopen(p, "rb"); if (!f){perror(p); exit(1);}
    void* b = malloc(n); size_t r = fread(b,1,n,f); (void)r; fclose(f); return b;
}

int main(int argc, char** argv) {
    if (argc != 2) return 1;
    int N = atoi(argv[1]);
    float* inp  = (float*) rd("course/ch05_build/inp.bin",  (size_t)N*sizeof(float));
    float* dout = (float*) rd("course/ch05_build/dout.bin", (size_t)N*sizeof(float));
    float* dinp = (float*) calloc((size_t)N, sizeof(float));
    gelu_backward(dinp, inp, dout, N);
    FILE* f = fopen("course/ch05_build/dinp.bin","wb"); fwrite(dinp,4,(size_t)N,f); fclose(f);
    free(inp); free(dout); free(dinp); return 0;
}


In [ ]:
!gcc -O2 -Wall -o course/ch05_build/gelu_backward course/ch05_build/gelu_backward.c -lm


In [ ]:
import numpy as np, torch, torch.nn as nn, subprocess
torch.manual_seed(0); N = 1000
x = (torch.randn(N) * 3.0).requires_grad_()   # leaf tensor with grad enabled
out = nn.GELU(approximate='tanh')(x)
dout = torch.randn_like(out)
out.backward(dout)

x.detach().numpy().astype(np.float32).tofile("course/ch05_build/inp.bin")
dout.numpy().astype(np.float32).tofile("course/ch05_build/dout.bin")
subprocess.run(["./course/ch05_build/gelu_backward", str(N)], check=True)
dinp_c = np.fromfile("course/ch05_build/dinp.bin", dtype=np.float32)
print(f"GELU backward max diff: {np.max(np.abs(dinp_c - x.grad.numpy())):.2e}")


## 8. Toy Example — GELU at a Few Points

Let's plot GELU and its derivative at hand-picked values so the saturation behavior is concrete.


In [ ]:
%%writefile course/ch05_build/toy_gelu.c
#include <stdio.h>
#include <math.h>
#define GELU_SCALING_FACTOR sqrtf(2.0f / M_PI)

float gelu(float x) {
    float cube = 0.044715f * x * x * x;
    return 0.5f * x * (1.0f + tanhf(GELU_SCALING_FACTOR * (x + cube)));
}

float gelu_deriv(float x) {
    float cube = 0.044715f * x * x * x;
    float a = GELU_SCALING_FACTOR * (x + cube);
    float t = tanhf(a);
    float c = coshf(a);
    float sech2 = 1.0f / (c * c);
    return 0.5f*(1.0f + t) + x*0.5f*sech2*GELU_SCALING_FACTOR*(1.0f + 3.0f*0.044715f*x*x);
}

int main(void) {
    printf("   x       GELU(x)    GELU'(x)\n");
    for (float x = -5.0f; x <= 5.0f; x += 1.0f) {
        printf("%+5.1f     %+8.4f   %+8.4f\n", x, gelu(x), gelu_deriv(x));
    }
    return 0;
}


In [ ]:
!gcc -O2 -Wall -o course/ch05_build/toy_gelu course/ch05_build/toy_gelu.c -lm && ./course/ch05_build/toy_gelu


You should see:

- For `x ≤ -3`: both `GELU(x)` and `GELU'(x)` are **near zero** — the function squashes large negatives to nothing, and there's almost no gradient flowing back. This is the GELU equivalent of ReLU's "dead zone", but smooth.
- For `x ≥ 3`: `GELU(x) ≈ x` and `GELU'(x) ≈ 1` — the function passes through with full gradient, like a linear identity.
- Around `x = 0`: smooth, with `GELU'(0) = 0.5`, **half the gradient passes through** at zero. (Compare to ReLU's discontinuous 0/1 derivative.)


## 9. Residual Connections — Concept

The Transformer block is

```
x → x + Attention(LayerNorm(x))
  → x + MLP(LayerNorm(x))
```

Each `x + f(x)` is a **residual connection**. Forward: just an elementwise add. Backward: the magic.

### Forward (4 lines of C)

```c
void residual_forward(float* out, float* inp1, float* inp2, int N) {
    for (int i = 0; i < N; i++) {
        out[i] = inp1[i] + inp2[i];
    }
}
```

### Backward — gradient splits, doesn't shrink

For $\text{out}_i = \text{inp1}_i + \text{inp2}_i$, the partials are simply:

$$\frac{\partial \text{out}_i}{\partial \text{inp1}_i} = 1 \qquad \frac{\partial \text{out}_i}{\partial \text{inp2}_i} = 1$$

So the upstream `dout[i]` flows to **both** branches **unchanged**:

```c
void residual_backward(float* dinp1, float* dinp2, float* dout, int N) {
    for (int i = 0; i < N; i++) {
        dinp1[i] += dout[i];
        dinp2[i] += dout[i];
    }
}
```

This is the whole reason residual networks train. The gradient at the **deep end** of the network always has a path back to the **shallow end** that is multiplied by **1.0 at every residual** — it doesn't get squashed by tanh, halved by GELU, or reshaped by anything else. Without residuals, gradients have to survive 12 (GPT-2 small) or 96 (GPT-3) blocks of `tanh` and `sech²` and matmul Jacobians, and they exponentially shrink. With residuals, there's an unobstructed gradient highway from the loss to every layer.

A neat way to see it: the network is computing $f_{12}(f_{11}(\dots f_1(x)\dots))$ but at every layer there's also an `+x` shortcut. The *effective* gradient of the loss with respect to layer 3's output has a direct (Jacobian = identity) path from layer 12's output, plus a long product through 4 → 5 → … → 12. The identity path dominates early in training, and it's why initializing all the residual branches near zero ("ReZero", "deep network init tricks") makes deep networks trainable.


## 10. Compile, Run, Verify Residual

In [ ]:
%%writefile course/ch05_build/residual.c
#include <stdio.h>
#include <stdlib.h>

void residual_forward(float* out, float* inp1, float* inp2, int N) {
    for (int i = 0; i < N; i++) out[i] = inp1[i] + inp2[i];
}

void residual_backward(float* dinp1, float* dinp2, float* dout, int N) {
    for (int i = 0; i < N; i++) {
        dinp1[i] += dout[i];
        dinp2[i] += dout[i];
    }
}

static void* rd(const char* p, size_t n) {
    FILE* f = fopen(p, "rb"); if (!f){perror(p); exit(1);}
    void* b = malloc(n); size_t r = fread(b,1,n,f); (void)r; fclose(f); return b;
}

int main(int argc, char** argv) {
    if (argc != 2) return 1;
    int N = atoi(argv[1]);
    float* inp1 = (float*) rd("course/ch05_build/inp1.bin", (size_t)N*sizeof(float));
    float* inp2 = (float*) rd("course/ch05_build/inp2.bin", (size_t)N*sizeof(float));
    float* dout = (float*) rd("course/ch05_build/dout.bin", (size_t)N*sizeof(float));
    float* out  = (float*) malloc((size_t)N*sizeof(float));
    float* dinp1 = (float*) calloc((size_t)N, sizeof(float));
    float* dinp2 = (float*) calloc((size_t)N, sizeof(float));
    residual_forward(out, inp1, inp2, N);
    residual_backward(dinp1, dinp2, dout, N);
    FILE* f;
    f=fopen("course/ch05_build/out.bin",  "wb"); fwrite(out,  4,(size_t)N,f); fclose(f);
    f=fopen("course/ch05_build/dinp1.bin","wb"); fwrite(dinp1,4,(size_t)N,f); fclose(f);
    f=fopen("course/ch05_build/dinp2.bin","wb"); fwrite(dinp2,4,(size_t)N,f); fclose(f);
    free(inp1); free(inp2); free(dout); free(out); free(dinp1); free(dinp2); return 0;
}


In [ ]:
!gcc -O3 -Wall -o course/ch05_build/residual course/ch05_build/residual.c


In [ ]:
import numpy as np, torch, subprocess
torch.manual_seed(0); N = 100
a = torch.randn(N, requires_grad=True)
b = torch.randn(N, requires_grad=True)
out = a + b
dout = torch.randn(N); out.backward(dout)

a.detach().numpy().astype(np.float32).tofile("course/ch05_build/inp1.bin")
b.detach().numpy().astype(np.float32).tofile("course/ch05_build/inp2.bin")
dout.numpy().astype(np.float32).tofile("course/ch05_build/dout.bin")
subprocess.run(["./course/ch05_build/residual", str(N)], check=True)

out_c   = np.fromfile("course/ch05_build/out.bin",   dtype=np.float32)
dinp1_c = np.fromfile("course/ch05_build/dinp1.bin", dtype=np.float32)
dinp2_c = np.fromfile("course/ch05_build/dinp2.bin", dtype=np.float32)
print(f"out   diff: {np.max(np.abs(out_c   - (a+b).detach().numpy())):.2e}")
print(f"dinp1 diff: {np.max(np.abs(dinp1_c - a.grad.numpy())):.2e}")
print(f"dinp2 diff: {np.max(np.abs(dinp2_c - b.grad.numpy())):.2e}")
print()
print("dout[0:5]    :", dout[:5].tolist())
print("dinp1[0:5]   :", dinp1_c[:5].tolist())
print("dinp2[0:5]   :", dinp2_c[:5].tolist())
print(" -> dinp1 == dinp2 == dout, exactly. The gradient was *split* (copied), not divided.")


## 11. Why Element-wise Ops Foreshadow GPU Programming

Notice how trivial both ops are in C:

- `gelu_forward`: 3 lines.
- `residual_forward`: 1 line.
- `residual_backward`: 2 lines.

There is **no `(b, t)` indexing**, **no shared state**, **no reductions**, **no reuse trick**. Every output element depends on at most a couple of input elements at the same flat index. Each iteration of the loop is *completely independent*.

This is the **ideal pattern for a GPU**: a CUDA kernel can launch one thread per element, and each thread does the same handful of FLOPs. There's no synchronization, no atomics, no shared memory needed. We'll meet our first such kernel in Chapter 9 (`gelu_forward.cu`) — and you'll see it's barely longer than the C version above, just with `threadIdx.x + blockIdx.x*blockDim.x` replacing the `for (int i = 0; i < N; i++)` line.

Element-wise ops are **bandwidth-bound** — the limit is how fast you can stream the input/output through memory, not how fast you can compute. For modern GPUs, peak GELU throughput is the same as peak `memcpy` throughput.


## 12. TODO Exercise 1 — Write `gelu_forward`

Just the forward — fill in the three TODOs.


In [ ]:
%%writefile course/ch05_build/exercise1.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

#define GELU_SCALING_FACTOR sqrtf(2.0f / M_PI)

void gelu_forward(float* out, float* inp, int N) {
    for (int i = 0; i < N; i++) {
        float x = inp[i];
        // TODO 1: compute cube = 0.044715 * x^3
        float cube = 0.0f;
        // TODO 2: compute the argument to tanh: GELU_SCALING_FACTOR * (x + cube)
        float arg = 0.0f;
        // TODO 3: out[i] = 0.5 * x * (1 + tanhf(arg))
        out[i] = 0.0f;
    }
}

static void* rd(const char* p, size_t n){FILE*f=fopen(p,"rb");void*b=malloc(n);size_t r=fread(b,1,n,f);(void)r;fclose(f);return b;}

int main(int argc, char** argv) {
    int N = atoi(argv[1]);
    float* inp = (float*) rd("course/ch05_build/inp.bin", (size_t)N*sizeof(float));
    float* out = (float*) malloc((size_t)N*sizeof(float));
    gelu_forward(out, inp, N);
    FILE* f = fopen("course/ch05_build/out_ex1.bin","wb"); fwrite(out,4,(size_t)N,f); fclose(f);
    free(inp); free(out); return 0;
}


In [ ]:
# Auto-grade Exercise 1
import numpy as np, torch, torch.nn as nn, subprocess
torch.manual_seed(0); N = 1000
x = torch.randn(N) * 3.0
x.numpy().astype(np.float32).tofile("course/ch05_build/inp.bin")
subprocess.run(["gcc","-O3","-Wall","-o","course/ch05_build/exercise1","course/ch05_build/exercise1.c","-lm"], check=True)
subprocess.run(["./course/ch05_build/exercise1", str(N)], check=True)
out_ex = np.fromfile("course/ch05_build/out_ex1.bin", dtype=np.float32)
out_pt = nn.GELU(approximate='tanh')(x).numpy()
err = np.max(np.abs(out_ex - out_pt))
print(f"max diff: {err:.2e}")
print("PASS" if err < 1e-5 else "FAIL — check the cube, the tanh argument, and the final formula")


### Solution to Exercise 1

In [ ]:
%%writefile course/ch05_build/exercise1_sol.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

#define GELU_SCALING_FACTOR sqrtf(2.0f / M_PI)

void gelu_forward(float* out, float* inp, int N) {
    for (int i = 0; i < N; i++) {
        float x = inp[i];
        float cube = 0.044715f * x * x * x;
        float arg  = GELU_SCALING_FACTOR * (x + cube);
        out[i] = 0.5f * x * (1.0f + tanhf(arg));
    }
}

static void* rd(const char* p, size_t n){FILE*f=fopen(p,"rb");void*b=malloc(n);size_t r=fread(b,1,n,f);(void)r;fclose(f);return b;}

int main(int argc, char** argv) {
    int N = atoi(argv[1]);
    float* inp = (float*) rd("course/ch05_build/inp.bin", (size_t)N*sizeof(float));
    float* out = (float*) malloc((size_t)N*sizeof(float));
    gelu_forward(out, inp, N);
    FILE* f = fopen("course/ch05_build/out_ex1.bin","wb"); fwrite(out,4,(size_t)N,f); fclose(f);
    free(inp); free(out); return 0;
}


In [ ]:
!gcc -O3 -Wall -o course/ch05_build/exercise1_sol course/ch05_build/exercise1_sol.c -lm && ./course/ch05_build/exercise1_sol 1000 && echo ran


## 13. TODO Exercise 2 — Residual Forward and Backward

Two of the simplest functions in the entire codebase. Fill in the four TODOs.


In [ ]:
%%writefile course/ch05_build/exercise2.c
#include <stdio.h>
#include <stdlib.h>

void residual_forward(float* out, float* inp1, float* inp2, int N) {
    for (int i = 0; i < N; i++) {
        // TODO 1: out[i] = inp1[i] + inp2[i];
    }
}

void residual_backward(float* dinp1, float* dinp2, float* dout, int N) {
    for (int i = 0; i < N; i++) {
        // TODO 2: accumulate dout[i] into dinp1[i]
        // TODO 3: accumulate dout[i] into dinp2[i]
    }
}

static void* rd(const char* p, size_t n){FILE*f=fopen(p,"rb");void*b=malloc(n);size_t r=fread(b,1,n,f);(void)r;fclose(f);return b;}

int main(int argc, char** argv) {
    int N = atoi(argv[1]);
    float* inp1 = (float*) rd("course/ch05_build/inp1.bin", (size_t)N*sizeof(float));
    float* inp2 = (float*) rd("course/ch05_build/inp2.bin", (size_t)N*sizeof(float));
    float* dout = (float*) rd("course/ch05_build/dout.bin", (size_t)N*sizeof(float));
    float* out   = (float*) calloc((size_t)N, sizeof(float));
    float* dinp1 = (float*) calloc((size_t)N, sizeof(float));
    float* dinp2 = (float*) calloc((size_t)N, sizeof(float));
    residual_forward(out, inp1, inp2, N);
    residual_backward(dinp1, dinp2, dout, N);
    FILE* f;
    f=fopen("course/ch05_build/out_ex2.bin",  "wb"); fwrite(out,  4,(size_t)N,f); fclose(f);
    f=fopen("course/ch05_build/dinp1_ex2.bin","wb"); fwrite(dinp1,4,(size_t)N,f); fclose(f);
    f=fopen("course/ch05_build/dinp2_ex2.bin","wb"); fwrite(dinp2,4,(size_t)N,f); fclose(f);
    free(inp1); free(inp2); free(dout); free(out); free(dinp1); free(dinp2); return 0;
}


In [ ]:
# Auto-grade Exercise 2
import numpy as np, torch, subprocess
torch.manual_seed(0); N = 100
a = torch.randn(N, requires_grad=True); b = torch.randn(N, requires_grad=True)
out = a + b; dout = torch.randn(N); out.backward(dout)
a.detach().numpy().astype(np.float32).tofile("course/ch05_build/inp1.bin")
b.detach().numpy().astype(np.float32).tofile("course/ch05_build/inp2.bin")
dout.numpy().astype(np.float32).tofile("course/ch05_build/dout.bin")
subprocess.run(["gcc","-O3","-Wall","-o","course/ch05_build/exercise2","course/ch05_build/exercise2.c"], check=True)
subprocess.run(["./course/ch05_build/exercise2", str(N)], check=True)
out_ex   = np.fromfile("course/ch05_build/out_ex2.bin",   dtype=np.float32)
dinp1_ex = np.fromfile("course/ch05_build/dinp1_ex2.bin", dtype=np.float32)
dinp2_ex = np.fromfile("course/ch05_build/dinp2_ex2.bin", dtype=np.float32)
e0 = np.max(np.abs(out_ex   - (a+b).detach().numpy()))
e1 = np.max(np.abs(dinp1_ex - a.grad.numpy()))
e2 = np.max(np.abs(dinp2_ex - b.grad.numpy()))
print(f"out  diff: {e0:.2e}\ndinp1 diff: {e1:.2e}\ndinp2 diff: {e2:.2e}")
print("PASS" if max(e0,e1,e2) < 1e-6 else "FAIL — check that BOTH dinp1 and dinp2 receive dout (unchanged)")


### Solution to Exercise 2

In [ ]:
%%writefile course/ch05_build/exercise2_sol.c
#include <stdio.h>
#include <stdlib.h>

void residual_forward(float* out, float* inp1, float* inp2, int N) {
    for (int i = 0; i < N; i++) out[i] = inp1[i] + inp2[i];
}

void residual_backward(float* dinp1, float* dinp2, float* dout, int N) {
    for (int i = 0; i < N; i++) {
        dinp1[i] += dout[i];
        dinp2[i] += dout[i];
    }
}

static void* rd(const char* p, size_t n){FILE*f=fopen(p,"rb");void*b=malloc(n);size_t r=fread(b,1,n,f);(void)r;fclose(f);return b;}

int main(int argc, char** argv) {
    int N = atoi(argv[1]);
    float* inp1 = (float*) rd("course/ch05_build/inp1.bin", (size_t)N*sizeof(float));
    float* inp2 = (float*) rd("course/ch05_build/inp2.bin", (size_t)N*sizeof(float));
    float* dout = (float*) rd("course/ch05_build/dout.bin", (size_t)N*sizeof(float));
    float* out   = (float*) calloc((size_t)N, sizeof(float));
    float* dinp1 = (float*) calloc((size_t)N, sizeof(float));
    float* dinp2 = (float*) calloc((size_t)N, sizeof(float));
    residual_forward(out, inp1, inp2, N);
    residual_backward(dinp1, dinp2, dout, N);
    FILE* f;
    f=fopen("course/ch05_build/out_ex2.bin",  "wb"); fwrite(out,  4,(size_t)N,f); fclose(f);
    f=fopen("course/ch05_build/dinp1_ex2.bin","wb"); fwrite(dinp1,4,(size_t)N,f); fclose(f);
    f=fopen("course/ch05_build/dinp2_ex2.bin","wb"); fwrite(dinp2,4,(size_t)N,f); fclose(f);
    free(inp1); free(inp2); free(dout); free(out); free(dinp1); free(dinp2); return 0;
}


In [ ]:
!gcc -O3 -Wall -o course/ch05_build/exercise2_sol course/ch05_build/exercise2_sol.c && ./course/ch05_build/exercise2_sol 100 && echo ran


## Recap

You now know:

- GPT-2 uses the **tanh approximation** of GELU. Loading GPT-2 weights and using the exact erf form would silently give wrong results — match the formula to the trained model.
- A 1-line forward can have a **5-term backward**. The product+chain rule gives `0.5(1+tanh) + 0.5x sech²(g) g'(x)`.
- The `-Ofast` / finite-math optimization is **incorrect** for `1/(cosh*cosh)` near saturation — `llm.c` patches around it with a per-function pragma.
- Residual backward **splits** the gradient: it copies `dout` to *both* `dinp1` and `dinp2`. That's the gradient highway that makes deep Transformers trainable.
- Element-wise ops have **no inter-element dependency** — the perfect mental model for what a GPU thread does.

### What's next

**Chapter 6 — Multi-Head Self-Attention.** The Transformer's signature operation, and the most expensive layer per FLOP-times-bytes-touched. We'll trace `attention_forward` end-to-end: how `(B, T, 3*C)` becomes `Q`, `K`, `V`; how the `(T, T)` attention matrix is computed and masked; how scaled softmax preserves probabilities; and how multi-head splitting changes nothing about the math — just how it's indexed. Then the backward, which is the single longest function in `train_gpt2.c`.

When you're ready, say **"proceed to Chapter 6"**.
